### Set up 

In [11]:
import pandas as pd
from pathlib import Path

# Définition du seuil académique de départ
start_date = pd.to_datetime('2015-03-01')

# Définition des régions d'intérêt
regions = ['US', 'France']

# Chargement de la base RÉGIONALE (indicators_geo)
dir_geo = Path("./indicators_geo_monthly") 
if dir_geo.exists():
    files_geo = list(dir_geo.glob("*.parquet"))
    df_geo = pd.concat([pd.read_parquet(f) for f in files_geo], ignore_index=True)
    df_geo = df_geo.groupby(['period', 'region_key']).max().reset_index()
    df_geo['period'] = pd.to_datetime(df_geo['period'])
    
    # Filtrage temporel et régional
    df_geo = df_geo[(df_geo['period'] >= start_date) & (df_geo['region_key'].isin(regions))]
    df_geo = df_geo.sort_values(['period', 'region_key'])
    
    print(f"✓ Base RÉGIONALE chargée : {len(df_geo)} lignes, avec les régions {list(df_geo['region_key'].unique())}")
    print(f"  Départ de la série : {df_geo['period'].min().date()}")
else:
    df_geo = None
    print("⚠ Dossier ./indicators_geo_monthly introuvable.")

# Note: L'ajout des données d'inflation, des lags et des variables macroéconomiques 
# devra être fait à cette étape pour construire la matrice X (variables explicatives) 
# et le vecteur y (variable à expliquer) pour chaque pays.

✓ Base RÉGIONALE chargée : 272 lignes, avec les régions ['France', 'US']
  Départ de la série : 2015-03-01


/tmp/ipykernel_3933455/3993766652.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_geo = df_geo.groupby(['period', 'region_key']).max().reset_index()
/tmp/ipykernel_3933455/3993766652.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_geo = df_geo.groupby(['period', 'region_key']).max().reset_index()


### GL and SGL - France

In [12]:
import pandas as pd
import numpy as np
import pandas_datareader.data as web
from datetime import datetime
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.preprocessing import StandardScaler

# =======================================================
# ÉTAPE 0 : RECRÉATION DE LA MACRO (Avec les lags longs)
# =======================================================
print("=== ÉTAPE 0 : TÉLÉCHARGEMENT DE LA MACRO ===")
start_date = datetime(2014, 11, 1)
end_date = datetime(2026, 8, 1)

tickers = {
    'France': {
        'cpi': 'CP0000FRM086NEST',     # IPCH France
        'rate': 'IRSTCI01FRM156N',     # Taux d'intérêt court terme
        'unemp': 'LRHUTTTTFRM156S',    # Taux de chômage
        'indpro': 'FRAPRINTO01GYSAM',   # Production Industrielle
        'money': 'EA19MABMM301GYSAM',  # M3 Euro Area (Déjà en taux de croissance)
        'fx': 'DEXUSEU',               # Taux de change USD/EUR
        'mich': 'CSINFT02FRM460S',     # Anticipations d'inflation
        'oil': 'DCOILBRENTEU'           # Prix du pétrole Brent
    }
}

def build_macro_dataframe(region):
    rename_map = {v: k for k, v in tickers[region].items()}
    region_tickers = list(tickers[region].values())
    
    df = web.DataReader(region_tickers, 'fred', start_date, end_date)
    df = df.rename(columns=rename_map)
    df = df.resample('MS').mean()
    
    # Variations en %
    df['inflation'] = df['cpi'].pct_change() * 100
    df['indpro_growth'] = df['indpro'].pct_change() * 100
    df['fx_growth'] = df['fx'].pct_change() * 100
    df['oil_growth'] = df['oil'].pct_change() * 100
    df['money_growth'] = df['money'] # La série Euro est déjà en %
    
    # Différences
    df['rate_diff'] = df['rate'].diff()
    df['unemp_diff'] = df['unemp'].diff()
    df['mich_diff'] = df['mich'].diff()
    
    # Lags d'inflation pour l'inertie
    df['inflation_lag1'] = df['inflation'].shift(1)
    df['inflation_lag2'] = df['inflation'].shift(2)
    df['inflation_lag3'] = df['inflation'].shift(3)
    df['inflation_lag12'] = df['inflation'].shift(12)
    
    df = df.dropna()
    df.index.name = 'period'
    df = df[df.index >= '2015-03-01']
    
    cols_to_keep = [
        'inflation', 'inflation_lag1', 'inflation_lag2', 'inflation_lag3', 'inflation_lag12',
        'rate_diff', 'unemp_diff', 'indpro_growth', 'money_growth', 'fx_growth', 'mich_diff', 'oil_growth'
    ]
    return df[cols_to_keep].copy()

df_macro_FR = build_macro_dataframe('France')
print("✓ Macro France téléchargée et prête.")

# =======================================================
# ÉTAPE 1 : PRÉPARATION GDELT ET JOINTURE CONTEMPORAINE
# =======================================================
print("\n=== ÉTAPE 1 : PRÉPARATION DES DONNÉES GDELT (FRANCE) ===")

cols_to_keep_gdelt = [col for col in df_geo.columns if str(col).startswith('att_weight_')]

df_FR = df_geo[df_geo['region_key'] == 'France'].sort_values('period').set_index('period')
X_gdelt_raw = df_FR[cols_to_keep_gdelt].copy()

X_gdelt_stat = X_gdelt_raw.copy()
non_stat_count = 0
for col in X_gdelt_stat.columns:
    if len(X_gdelt_stat[col].dropna()) > 10:
        if adfuller(X_gdelt_stat[col].dropna(), autolag='AIC')[1] >= 0.05:
            X_gdelt_stat[col] = X_gdelt_stat[col].diff()
            non_stat_count += 1
X_gdelt_stat = X_gdelt_stat.dropna()
print(f"✓ {non_stat_count} variables GDELT différenciées pour stationnarité.")

# Jointure CONTEMPORAINE (pas de décalage temporel entre GDELT et l'inflation)
df_final = X_gdelt_stat.join(df_macro_FR, how='inner')

y_raw = df_final['inflation']
X_full = df_final.drop(columns=['inflation'])

# Désaisonnalisation spécifique France
decomposition = seasonal_decompose(y_raw, model='additive', period=12)
y_corr = (y_raw - decomposition.seasonal).dropna()

# Alignement final
y_corr, X_contemporaneous = y_corr.align(X_full, join='inner')

# Standardisation
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X_contemporaneous), 
    columns=X_contemporaneous.columns, 
    index=X_contemporaneous.index
)

print(f"✓ Matrice finale alignée en t : {X_scaled.shape[0]} mois, {X_scaled.shape[1]} variables explicatives.")

=== ÉTAPE 0 : TÉLÉCHARGEMENT DE LA MACRO ===


/tmp/ipykernel_3933455/2025353301.py:33: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  df = web.DataReader(region_tickers, 'fred', start_date, end_date)


✓ Macro France téléchargée et prête.

=== ÉTAPE 1 : PRÉPARATION DES DONNÉES GDELT (FRANCE) ===
✓ 26 variables GDELT différenciées pour stationnarité.
✓ Matrice finale alignée en t : 110 mois, 70 variables explicatives.


In [13]:
from sklearn.linear_model import LinearRegression
from statsmodels.stats.diagnostic import acorr_ljungbox

print("=== ÉTAPE 2 : FILTRAGE MACROÉCONOMIQUE (FWL) - FRANCE ===")

# 1. Séparation explicite des blocs Macro et GDELT
macro_cols = [
    'inflation_lag1', 'inflation_lag2', 'inflation_lag3', 'inflation_lag12',
    'rate_diff', 'unemp_diff', 'indpro_growth', 'money_growth', 'fx_growth', 'mich_diff', 'oil_growth'
]
gdelt_cols = [col for col in X_scaled.columns if str(col).startswith('att_weight_')]

X_macro = X_scaled[macro_cols]
X_gdelt = X_scaled[gdelt_cols]

# 2. Régression OLS de l'inflation sur la Macro
print("[1/2] Purge de l'effet macro sur l'inflation...")
ols_y = LinearRegression(fit_intercept=True)
ols_y.fit(X_macro, y_corr)
# On récupère les résidus : ce que la macro n'arrive pas à expliquer (le "choc d'inflation")
y_tilde = y_corr - ols_y.predict(X_macro)

# 3. Régression OLS de chaque variable GDELT sur la Macro
print("[2/2] Purge de l'effet macro sur les narratifs GDELT...")
ols_X = LinearRegression(fit_intercept=True)
ols_X.fit(X_macro, X_gdelt)
# On récupère les résidus : la variation des médias indépendante de la conjoncture économique pure
X_gdelt_tilde = pd.DataFrame(
    X_gdelt.values - ols_X.predict(X_macro), 
    columns=X_gdelt.columns, 
    index=X_gdelt.index
)

# 4. Test d'Autocorrélation (Sanity Check)
# Si notre bloc macro (avec les lags 1, 2, 3 et 12) a bien fait son travail, 
# y_tilde ne devrait plus avoir de "mémoire" (il doit être un bruit blanc contemporain).
lb_test = acorr_ljungbox(y_tilde, lags=[12], return_df=True)
p_value_lb = lb_test['lb_pvalue'].values[0]

print("\n=== Diagnostic des Résidus de l'Inflation (y_tilde) ===")
if p_value_lb < 0.05:
    print(f"⚠ ATTENTION : Il reste de l'autocorrélation (p-value = {p_value_lb:.4f}).")
else:
    print(f"✓ OK : Succès total. La dynamique temporelle a été absorbée par la macro (p-value = {p_value_lb:.4f}).")
    print("  -> Nous pouvons maintenant analyser les co-mouvements purs.")

print(f"\n✓ Matrice GDELT purifiée prête : {X_gdelt_tilde.shape[0]} mois, {X_gdelt_tilde.shape[1]} variables.")

=== ÉTAPE 2 : FILTRAGE MACROÉCONOMIQUE (FWL) - FRANCE ===
[1/2] Purge de l'effet macro sur l'inflation...
[2/2] Purge de l'effet macro sur les narratifs GDELT...

=== Diagnostic des Résidus de l'Inflation (y_tilde) ===
✓ OK : Succès total. La dynamique temporelle a été absorbée par la macro (p-value = 0.2323).
  -> Nous pouvons maintenant analyser les co-mouvements purs.

✓ Matrice GDELT purifiée prête : 110 mois, 59 variables.


In [18]:
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from group_lasso import GroupLasso
import numpy as np
import pandas as pd

print("=== ÉTAPE 3 (ALTERNATIVE) : GROUP LASSO PUR - FRANCE ===")

# 1. Recréation propre des groupes pour la France
secteurs_fr = [col.split('_')[2] for col in X_gdelt_tilde.columns]
unique_secteurs_fr = list(set(secteurs_fr))
sector_to_id_fr = {sec: i for i, sec in enumerate(unique_secteurs_fr)}
groups_gdelt_fr = np.array([sector_to_id_fr[sec] for sec in secteurs_fr])

print("Secteurs soumis au Group Lasso pur :")
for sec, gid in sector_to_id_fr.items():
    print(f"  - {sec.capitalize()} (ID {gid})")

# 2. Configuration de la grille pour un Group Lasso pur (l1_reg = 0 fixe)
param_grid_gl = {
    'group_reg': np.logspace(-3, 1, 20)
}
tscv = TimeSeriesSplit(n_splits=5)

gl_pure = GroupLasso(
    groups=groups_gdelt_fr,
    l1_reg=0.0,              # AUCUNE pénalité individuelle : si un secteur est sélectionné, toutes ses variables le sont !
    scale_reg='group_size',
    fit_intercept=True,
    n_iter=10000,
    tol=1e-4,
    supress_warning=True
)

# 3. Lancement de la recherche du Lambda optimal
print("\nRecherche du paramètre optimal de groupe (Group Lasso pur)...")
grid_search_gl = GridSearchCV(
    estimator=gl_pure,
    param_grid=param_grid_gl,
    cv=tscv,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

grid_search_gl.fit(X_gdelt_tilde.values, y_tilde.values.reshape(-1, 1))
best_gl = grid_search_gl.best_estimator_

print(f"✓ Lambda optimal (group_reg) : {grid_search_gl.best_params_['group_reg']:.4f}")

# 4. Extraction et Affichage des Résultats par Secteur
coef_df_gl = pd.DataFrame({
    'Variable': X_gdelt_tilde.columns,
    'Secteur': secteurs_fr,
    'Coefficient': best_gl.coef_.flatten()
})

# On filtre les secteurs qui ont un coefficient non nul
active_variables_gl = coef_df_gl[coef_df_gl['Coefficient'] != 0]

print("\n=== SECTEURS CONSERVÉS PAR LE GROUP LASSO PUR (FRANCE) ===")
if active_variables_gl.empty:
    print("Le Group Lasso a tout mis à zéro : Aucun secteur GDELT ne co-varie avec l'inflation purifiée.")
else:
    print(active_variables_gl[['Secteur', 'Variable', 'Coefficient']].to_string(index=False))

=== ÉTAPE 3 (ALTERNATIVE) : GROUP LASSO PUR - FRANCE ===
Secteurs soumis au Group Lasso pur :
  - Energy (ID 0)
  - Transport (ID 1)
  - Industry (ID 2)
  - Commodities (ID 3)
  - Real (ID 4)
  - Agriculture (ID 5)
  - Finance (ID 6)
  - Tech (ID 7)

Recherche du paramètre optimal de groupe (Group Lasso pur)...


✓ Lambda optimal (group_reg) : 0.0183

=== SECTEURS CONSERVÉS PAR LE GROUP LASSO PUR (FRANCE) ===
Secteur                              Variable  Coefficient
finance                    att_weight_finance    -0.000279
finance              att_weight_finance_banks    -0.000183
finance      att_weight_finance_central_banks    -0.000188
finance             att_weight_finance_credit     0.000237
finance  att_weight_finance_financial_markets    -0.000118
finance att_weight_finance_international_orgs    -0.000232
finance       att_weight_finance_private_debt    -0.000576
finance         att_weight_finance_regulation    -0.000176
finance      att_weight_finance_systemic_risk    -0.000473
finance     att_weight_finance_sovereign_debt    -0.000191
   real                att_weight_real_estate    -0.006750
   real   att_weight_real_estate_construction    -0.001945
   real      att_weight_real_estate_financing    -0.005326
   real         att_weight_real_estate_market    -0.003558
   real     att_w

In [19]:
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from group_lasso import GroupLasso
import numpy as np
import pandas as pd

print("=== ÉTAPE 3 : SPARSE GROUP LASSO (CO-MOUVEMENTS) - FRANCE ===")

# 1. Création dynamique des groupes à partir du nom des colonnes
# (On extrait le 3e mot, ex: 'att' -> 'weight' -> 'SECTEUR')
secteurs = [col.split('_')[2] for col in X_gdelt_tilde.columns]
unique_secteurs = list(set(secteurs))

# On génère un dictionnaire qui associe chaque secteur à un ID (0, 1, 2...)
sector_to_id = {sec: i for i, sec in enumerate(unique_secteurs)}
groups_gdelt = np.array([sector_to_id[sec] for sec in secteurs])

print("Secteurs identifiés pour la pénalisation de groupe :")
for sec, gid in sector_to_id.items():
    print(f"  - {sec.capitalize()} (ID {gid})")

# 2. Configuration de la Validation Croisée et du Modèle
# On utilise une grille large, allant jusqu'à 10^-4 pour ne pas "étouffer" les signaux subtils
param_grid_sgl = {
    'group_reg': np.logspace(-4, 0, 10),
    'l1_reg': np.logspace(-4, 0, 10)
}
tscv = TimeSeriesSplit(n_splits=5)

sgl_base = GroupLasso(
    groups=groups_gdelt,
    scale_reg='group_size',
    fit_intercept=True,
    n_iter=10000,
    tol=1e-4,
    supress_warning=True
)

# 3. Lancement de la grille (GridSearchCV) sur les séries PURIFIÉES
print("\nRecherche de la double pénalité optimale (cela peut prendre quelques secondes)...")
grid_search_sgl = GridSearchCV(
    estimator=sgl_base,
    param_grid=param_grid_sgl,
    cv=tscv,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

grid_search_sgl.fit(X_gdelt_tilde.values, y_tilde.values.reshape(-1, 1))
best_sgl = grid_search_sgl.best_estimator_

print(f"✓ Configuration optimale : group_reg = {grid_search_sgl.best_params_['group_reg']:.4f}, l1_reg = {grid_search_sgl.best_params_['l1_reg']:.4f}")

# 4. Extraction et Affichage des Résultats
coef_df_sgl = pd.DataFrame({
    'Variable': X_gdelt_tilde.columns,
    'Secteur': secteurs,
    'Coefficient': best_sgl.coef_.flatten()
})

active_variables_sgl = coef_df_sgl[coef_df_sgl['Coefficient'] != 0].sort_values(
    by=['Secteur', 'Coefficient'], ascending=[True, False]
)

print("\n=== CO-MOUVEMENTS GDELT / CHOCS D'INFLATION (FRANCE) ===")
if active_variables_sgl.empty:
    print("Le SGL a tout mis à zéro : Les variables macro suffisent à expliquer l'inflation en t.")
else:
    print(active_variables_sgl.to_string(index=False))

=== ÉTAPE 3 : SPARSE GROUP LASSO (CO-MOUVEMENTS) - FRANCE ===
Secteurs identifiés pour la pénalisation de groupe :
  - Energy (ID 0)
  - Transport (ID 1)
  - Industry (ID 2)
  - Commodities (ID 3)
  - Real (ID 4)
  - Agriculture (ID 5)
  - Finance (ID 6)
  - Tech (ID 7)

Recherche de la double pénalité optimale (cela peut prendre quelques secondes)...
✓ Configuration optimale : group_reg = 0.0167, l1_reg = 0.0001

=== CO-MOUVEMENTS GDELT / CHOCS D'INFLATION (FRANCE) ===
                             Variable Secteur  Coefficient
            att_weight_finance_credit finance     0.001287
 att_weight_finance_financial_markets finance    -0.000572
    att_weight_finance_sovereign_debt finance    -0.000885
        att_weight_finance_regulation finance    -0.000888
             att_weight_finance_banks finance    -0.000916
     att_weight_finance_central_banks finance    -0.000957
att_weight_finance_international_orgs finance    -0.001204
                   att_weight_finance finance    -0.0

### GL and SGL - US

In [20]:
import pandas as pd
import numpy as np
import pandas_datareader.data as web
from datetime import datetime
from statsmodels.tsa.stattools import adfuller
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from group_lasso import GroupLasso

print("=== PIPELINE COMPTEUR : ÉTATS-UNIS ===")

start_date = datetime(2014, 11, 1)
end_date = datetime(2026, 8, 1)

# =======================================================
# ÉTAPE 0 : TÉLÉCHARGEMENT DE LA MACRO US
# =======================================================
tickers_us = {
    'US': {
        'cpi': 'CPIAUCSL',             # Inflation (CPI)
        'rate': 'FEDFUNDS',            # Taux d'intérêt court terme
        'unemp': 'UNRATE',             # Taux de chômage
        'indpro': 'INDPRO',            # Production Industrielle
        'money': 'M2SL',               # M2 US (Niveau)
        'fx': 'DEXUSEU',               # Taux de change USD/EUR
        'mich': 'MICH',                # Anticipations d'inflation (Michigan)
        'oil': 'WTISPLC'               # Prix du pétrole WTI
    }
}

rename_map_us = {v: k for k, v in tickers_us['US'].items()}
us_tickers_list = list(tickers_us['US'].values())

df_macro_US = web.DataReader(us_tickers_list, 'fred', start_date, end_date)
df_macro_US = df_macro_US.rename(columns=rename_map_us)
df_macro_US = df_macro_US.resample('MS').mean()

# Transformations
df_macro_US['inflation'] = df_macro_US['cpi'].pct_change() * 100
df_macro_US['indpro_growth'] = df_macro_US['indpro'].pct_change() * 100
df_macro_US['fx_growth'] = df_macro_US['fx'].pct_change() * 100
df_macro_US['oil_growth'] = df_macro_US['oil'].pct_change() * 100
df_macro_US['money_growth'] = df_macro_US['money'].pct_change() * 100

df_macro_US['rate_diff'] = df_macro_US['rate'].diff()
df_macro_US['unemp_diff'] = df_macro_US['unemp'].diff()
df_macro_US['mich_diff'] = df_macro_US['mich'].diff()

# Lags d'inflation US
df_macro_US['inflation_lag1'] = df_macro_US['inflation'].shift(1)
df_macro_US['inflation_lag2'] = df_macro_US['inflation'].shift(2)
df_macro_US['inflation_lag3'] = df_macro_US['inflation'].shift(3)
# df_macro_US['inflation_lag12'] = df_macro_US['inflation'].shift(12)

df_macro_US = df_macro_US.dropna()
df_macro_US.index.name = 'period'
df_macro_US = df_macro_US[df_macro_US.index >= '2015-03-01']

cols_macro_us = [
    'inflation', 'inflation_lag1', 'inflation_lag2', 'inflation_lag3', 
    'rate_diff', 'unemp_diff', 'indpro_growth', 'money_growth', 'fx_growth', 'mich_diff', 'oil_growth'
]
df_macro_US = df_macro_US[cols_macro_us].copy()
print("✓ Macro US téléchargée.")

# =======================================================
# ÉTAPE 1 : PRÉPARATION GDELT ET JOINTURE CONTEMPORAINE (US)
# =======================================================
cols_to_keep_gdelt = [col for col in df_geo.columns if str(col).startswith('att_weight_')]

df_US = df_geo[df_geo['region_key'] == 'US'].sort_values('period').set_index('period')
X_gdelt_raw = df_US[cols_to_keep_gdelt].copy()

X_gdelt_stat = X_gdelt_raw.copy()
for col in X_gdelt_stat.columns:
    if len(X_gdelt_stat[col].dropna()) > 10:
        if adfuller(X_gdelt_stat[col].dropna(), autolag='AIC')[1] >= 0.05:
            X_gdelt_stat[col] = X_gdelt_stat[col].diff()
            
X_gdelt_stat = X_gdelt_stat.dropna()

# Jointure contemporaine
df_final_us = X_gdelt_stat.join(df_macro_US, how='inner')

y_raw_us = df_final_us['inflation'] # Déjà désaisonnalisé à la source (CPIAUCSL)
X_full_us = df_final_us.drop(columns=['inflation'])

y_corr_us, X_contemp_us = y_raw_us.align(X_full_us, join='inner')

scaler_us = StandardScaler()
X_scaled_us = pd.DataFrame(
    scaler_us.fit_transform(X_contemp_us), 
    columns=X_contemp_us.columns, 
    index=X_contemp_us.index
)
print(f"✓ Matrice US prête en t : {X_scaled_us.shape[0]} mois, {X_scaled_us.shape[1]} variables.")

# =======================================================
# ÉTAPE 2 : FILTRAGE MACROÉCONOMIQUE (FWL) - US
# =======================================================
macro_cols_us = cols_macro_us[1:] # On exclut 'inflation'
gdelt_cols_us = [col for col in X_scaled_us.columns if str(col).startswith('att_weight_')]

X_macro_us = X_scaled_us[macro_cols_us]
X_gdelt_us = X_scaled_us[gdelt_cols_us]

# Purge OLS
ols_y_us = LinearRegression(fit_intercept=True).fit(X_macro_us, y_corr_us)
y_tilde_us = y_corr_us - ols_y_us.predict(X_macro_us)

ols_X_us = LinearRegression(fit_intercept=True).fit(X_macro_us, X_gdelt_us)
X_gdelt_tilde_us = pd.DataFrame(
    X_gdelt_us.values - ols_X_us.predict(X_macro_us), 
    columns=X_gdelt_us.columns, 
    index=X_gdelt_us.index
)

# Test Ljung-Box sur les résidus US
lb_test_us = acorr_ljungbox(y_tilde_us, lags=[12], return_df=True)
p_val_us = lb_test_us['lb_pvalue'].values[0]
print(f"Test Ljung-Box US p-value : {p_val_us:.4f} (Doit être > 0.05)")

# =======================================================
# ÉTAPE 3 : SPARSE GROUP LASSO (CO-MOUVEMENTS) - US
# =======================================================
secteurs_us = [col.split('_')[2] for col in X_gdelt_tilde_us.columns]
unique_secteurs_us = list(set(secteurs_us))
sector_to_id_us = {sec: i for i, sec in enumerate(unique_secteurs_us)}
groups_gdelt_us = np.array([sector_to_id_us[sec] for sec in secteurs_us])

param_grid_sgl = {'group_reg': np.logspace(-4, 0, 10), 'l1_reg': np.logspace(-4, 0, 10)}
tscv = TimeSeriesSplit(n_splits=5)

sgl_base_us = GroupLasso(
    groups=groups_gdelt_us, scale_reg='group_size', 
    fit_intercept=True, n_iter=10000, tol=1e-4, supress_warning=True
)

grid_sgl_us = GridSearchCV(sgl_base_us, param_grid=param_grid_sgl, cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1)
grid_sgl_us.fit(X_gdelt_tilde_us.values, y_tilde_us.values.reshape(-1, 1))
best_sgl_us = grid_sgl_us.best_estimator_

coef_df_us = pd.DataFrame({
    'Variable': X_gdelt_tilde_us.columns,
    'Secteur': secteurs_us,
    'Coefficient': best_sgl_us.coef_.flatten()
})

active_vars_us = coef_df_us[coef_df_us['Coefficient'] != 0].sort_values(by=['Secteur', 'Coefficient'], ascending=[True, False])

print("\n=== CO-MOUVEMENTS GDELT / CHOCS D'INFLATION (ÉTATS-UNIS) ===")
print(active_vars_us.to_string(index=False) if not active_vars_us.empty else "Aucune variable conservée.")

=== PIPELINE COMPTEUR : ÉTATS-UNIS ===


/tmp/ipykernel_3933455/3453588898.py:36: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  df_macro_US = web.DataReader(us_tickers_list, 'fred', start_date, end_date)


✓ Macro US téléchargée.
✓ Matrice US prête en t : 130 mois, 69 variables.
Test Ljung-Box US p-value : 0.2442 (Doit être > 0.05)

=== CO-MOUVEMENTS GDELT / CHOCS D'INFLATION (ÉTATS-UNIS) ===
                                  Variable     Secteur  Coefficient
att_weight_agriculture_infrastructure_tech agriculture    -0.002532
      att_weight_agriculture_climate_risks agriculture    -0.009641
      att_weight_commodities_market_prices commodities     0.055966
              att_weight_commodities_risks commodities    -0.001093
              att_weight_energy_renewables      energy     0.016767
                         att_weight_energy      energy     0.005060
         att_weight_finance_sovereign_debt     finance     0.009380
      att_weight_finance_financial_markets     finance    -0.010666
         att_weight_industry_supply_chains    industry    -0.015861
         att_weight_real_estate_regulation        real    -0.002457


In [21]:
print("=== ÉTAPE 3 (ALTERNATIVE) : GROUP LASSO PUR - ÉTATS-UNIS ===")

param_grid_gl_us = {'group_reg': np.logspace(-3, 1, 20)}

gl_pure_us = GroupLasso(
    groups=groups_gdelt_us,
    l1_reg=0.0,              # Group Lasso pur
    scale_reg='group_size',
    fit_intercept=True,
    n_iter=10000,
    tol=1e-4,
    supress_warning=True
)

grid_search_gl_us = GridSearchCV(
    estimator=gl_pure_us,
    param_grid=param_grid_gl_us,
    cv=tscv,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

grid_search_gl_us.fit(X_gdelt_tilde_us.values, y_tilde_us.values.reshape(-1, 1))
best_gl_us = grid_search_gl_us.best_estimator_

coef_df_gl_us = pd.DataFrame({
    'Variable': X_gdelt_tilde_us.columns,
    'Secteur': secteurs_us,
    'Coefficient': best_gl_us.coef_.flatten()
})

active_variables_gl_us = coef_df_gl_us[coef_df_gl_us['Coefficient'] != 0]

print("\n=== SECTEURS CONSERVÉS PAR LE GROUP LASSO PUR (ÉTATS-UNIS) ===")
if active_variables_gl_us.empty:
    print("✓ Confirmation : Aucun secteur GDELT ne co-varie avec l'inflation américaine une fois la macro contrôlée.")
else:
    print(active_variables_gl_us[['Secteur', 'Variable', 'Coefficient']].to_string(index=False))

=== ÉTAPE 3 (ALTERNATIVE) : GROUP LASSO PUR - ÉTATS-UNIS ===

=== SECTEURS CONSERVÉS PAR LE GROUP LASSO PUR (ÉTATS-UNIS) ===
    Secteur                             Variable  Coefficient
commodities               att_weight_commodities     0.005988
commodities    att_weight_commodities_extraction     0.001495
commodities att_weight_commodities_market_prices     0.033555
commodities     att_weight_commodities_materials    -0.000665
commodities    att_weight_commodities_regulation    -0.006032
commodities         att_weight_commodities_risks    -0.006957
